# Collaborative filtering recommender using Non-Negative Matrix Factorization

In the previous notebook we used kNN to cluster the user-item interactions. However, kNN is a memory based method, which means that it does not scale well with increasing input data size. This leads us to using the dimensionality reduction method NNMF, which reduces a large, sparse matrix into two smaller dense matrices.

The intermediate size of the factorized matrices is chosen as a hyperparameter. This way the size can be reduced from (j, k), to (j, i) @ (i, k). This results in a size of (ji+ik)=i(j+k) vs (jk). Choosing i smaller than j or k will reduce the dimension and modify the input size from multiplicative to additive.

This factorization method is a machine learning problem and is approached as such by setting a  costr function that can be optimized with SGD. The cost function is usually defined as:

$$\sum_{r_{jk} \in {train}} \left(r_{jk} - \hat{r}_{jk} \right)^2$$


In [ ]:
import pandas as pd

from surprise import NMF
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy

First, we'll load the dataset and convert it to a sparse format.

In [ ]:
rating_df = pd.read_csv("./data/ratings.csv")
rating_df.head()

,user,item,rating
0,1889878,CC0101EN,5
1,1342067,CL0101EN,3
2,1990814,ML0120ENv3,5
3,380098,BD0211EN,5
4,779563,DS0101EN,3


In [ ]:
rating_sparse_df = rating_df.pivot(index='user', columns='item', values='rating').fillna(0).reset_index().rename_axis(index=None, columns=None)
rating_sparse_df.head()

,user,AI0111EN,BC0101EN,BC0201EN,BC0202EN,BD0101EN,BD0111EN,BD0115EN,BD0121EN,BD0123EN,...,SW0201EN,TA0105,TA0105EN,TA0106EN,TMP0101EN,TMP0105EN,TMP0106,TMP107,WA0101EN,WA0103EN
0,2,0.0,4.0,0.0,0.0,5.0,4.0,0.0,5.0,3.0,...,0.0,5.0,0.0,4.0,0.0,3.0,3.0,0.0,5.0,0.0
1,4,0.0,0.0,0.0,0.0,5.0,3.0,4.0,5.0,3.0,...,0.0,4.0,0.0,0.0,0.0,3.0,3.0,0.0,3.0,3.0
2,5,3.0,5.0,5.0,0.0,4.0,0.0,0.0,0.0,3.0,...,0.0,0.0,4.0,4.0,4.0,4.0,4.0,5.0,0.0,3.0
3,7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,8,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Then we can use the surprise library to fit an NNMF model to the data

In [ ]:
# Save the rating dataframe to a CSV file
rating_df.to_csv("./data/course_ratings.csv", index=False)

# Read the course rating dataset with columns user item rating
reader = Reader(line_format='user item rating', sep=',', skip_lines=1, rating_scale=(2, 3))

# Load the dataset from the CSV file
course_dataset = Dataset.load_from_file("./data/course_ratings.csv", reader=reader)

trainset, testset = train_test_split(course_dataset, test_size=.3)
print(f"Total {trainset.n_users} users and {trainset.n_items} items in the trainingset")

Total 31318 users and 125 items in the trainingset


In [ ]:
nmf_model = NMF()
nmf_model.fit(trainset)
preds = nmf_model.test(testset)
accuracy.rmse(preds)

RMSE: 1.3002


1.3002114572611903